# Appendix — PydanticAI: the typed agent loop

Seven framework appendices run the *same* Larkspur triage — ticket `TKT-2205`, gold `partial_refund | pol-restocking | $170.99` — so you can compare frameworks on one fixed problem. This is the **typed** archetype: [Pydantic AI](https://pydantic.dev/docs/ai/overview/) is "a typed, extensible agent loop with every model a string swap away," and its distinguishing move is that the agent's answer comes back as a *validated Pydantic model*, not a string you parse. That is your chapter 01 structured-output boundary and your chapter 02 tool loop, wearing types.

> **Before running this notebook:** `pip install -e ".[pydantic-ai]"` (once). It pulls in `pydantic-ai-slim[openai]`, which co-installs with the main venv — no separate kernel. The model reaches the same OpenRouter endpoint you have used since ch01, through Pydantic AI's own OpenAI-compatible client. Everything else stays the same.

In [ ]:
# === config (identical in every notebook) ===
import os, getpass
import litellm
from dotenv import load_dotenv              # pip install -e ".[obs]" if this fails

load_dotenv(".env")   # reads OPENROUTER_API_KEY / MODEL / STRONG_MODEL (see .env.example)

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")

MODEL = os.environ.get("MODEL", "openrouter/deepseek/deepseek-v3.2")
STRONG_MODEL = os.environ.get("STRONG_MODEL", "openrouter/deepseek/deepseek-v4-flash")

# Per-notebook override: uncomment to ignore .env here (any LiteLLM provider works).
# MODEL = "openrouter/google/gemini-2.5-flash-lite"
# MODEL = "openai/gpt-4o-mini"              # direct OpenAI, uses OPENAI_API_KEY instead

TEMPERATURE = 0                             # the whole course runs at temperature 0
litellm.drop_params = True                  # ignore params a provider does not support
litellm.cache = litellm.Cache(type="disk", disk_cache_dir=".litellm_cache")  # reruns are ~free

## The model boundary, as a string swap

Pydantic AI reaches the model through a **provider + model class**, not `shoplab.llm.complete` or LiteLLM. We point an `OpenAIChatModel` at OpenRouter with the built-in `OpenRouterProvider` — the same OpenAI-compatible endpoint ch16 used, so the model id drops the `openrouter/` routing prefix and is just `deepseek/deepseek-v3.2`. We set `temperature=0` on the model to keep the course's determinism. One consequence worth repeating: these calls go through Pydantic AI's own client, so they skip the LiteLLM disk cache — a rerun costs the same cent or two.

In [ ]:
import os
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openrouter import OpenRouterProvider
from pydantic_ai.settings import ModelSettings

model = OpenAIChatModel(
    "deepseek/deepseek-v3.2",
    provider=OpenRouterProvider(api_key=os.environ["OPENROUTER_API_KEY"]),
    settings=ModelSettings(temperature=TEMPERATURE),   # the whole course runs at temp 0
)
print("model:", type(model).__module__ + "." + type(model).__name__)
print("model_name:", model.model_name)

## The decision as a typed schema

This is the row that names the archetype. In chapter 01 you built `parse_json_loose` to coax a dict out of the model's text and then checked its keys and types by hand. Pydantic AI turns that boundary inside out: you *declare* the shape as a `BaseModel` and hand it to the agent as `output_type`; the framework makes the model fill exactly that shape and validates it for you. The verdict comes back as a `Decision` instance, not a string.

Every appendix must land on the same economics, so pin the invariant now — the authoritative gold from `shoplab.rules.decide` for `TKT-2205` (an opened, in-window return from a non-vip member: `$189.99 x 0.90`).

In [ ]:
from typing import Literal, Optional
from pydantic import BaseModel
from shoplab import world, rules

class Decision(BaseModel):
    """The ops-desk verdict -- the agent must fill exactly this shape."""
    decision: Literal["approve_refund", "partial_refund", "replacement",
                      "store_credit", "deny", "escalate"]
    policy_id: str
    refund_usd: Optional[float] = None

orders = {o["order_id"]: o for o in world.load_orders()}
customers = {c["customer_id"]: c for c in world.load_customers()}
t2205 = next(t for t in world.load_tickets()["train"] if t["ticket_id"] == "TKT-2205")
GOLD = rules.decide(t2205, orders[t2205["order_id"]], customers[t2205["customer_id"]])
print("authoritative gold (shoplab.rules.decide):", GOLD)   # the invariant every appendix hits

## Tools: thin wrappers over shoplab

The graph needs the same Larkspur lookups the ops desk always uses. Pydantic AI's `@agent.tool_plain` decorator turns a plain annotated function into a tool — the docstring becomes the description, the type hints become the JSON schema — which is exactly what `shoplab.tools.to_openai_tools` did by hand in chapter 02. The four read-only lookups (`get_order`, `get_customer`, `search_policy`, `calc`) are thin wrappers that *import* from `shoplab.world` / `shoplab.tools`, not reimplementations. The risky `issue_refund` is registered with `requires_approval=True` — the flag that turns it into the approval gate of the next section.

In [ ]:
from pydantic_ai import Agent, DeferredToolRequests
from shoplab.tools import calc as _calc, Ledger

ledger = Ledger()                              # ch02's real side-effect log
agent = Agent(model, output_type=[Decision, DeferredToolRequests], instructions=(
    "You are the Larkspur ops desk. Triage the return ticket: look up the order and "
    "customer, search the governing policy, compute any restocking fee with calc, then "
    "call issue_refund to move the money, and finally return the Decision. Opened "
    "non-vip returns carry a 10% restocking fee."))

@agent.tool_plain
def get_order(order_id: str) -> dict:
    """Look up a Larkspur order by id (items, totals, status, dates)."""
    return orders.get(order_id, {"error": f"no such order {order_id}"})

@agent.tool_plain
def get_customer(customer_id: str) -> dict:
    """Look up a Larkspur customer by id (tier, flags, history)."""
    return customers.get(customer_id, {"error": f"no such customer {customer_id}"})

@agent.tool_plain
def search_policy(query: str, k: int = 2) -> list:
    """Keyword-search the 12 Larkspur store policy documents."""
    return world.search_policy(query, k=k)

@agent.tool_plain
def calc(expr: str) -> float:
    """Evaluate an arithmetic expression, e.g. '0.9 * 189.99'."""
    return _calc(expr)

@agent.tool_plain(requires_approval=True)          # the ch08 gate, one flag
def issue_refund(order_id: str, amount_usd: float, reason: str) -> dict:
    """Send money back to the customer. Irreversible."""
    entry = ledger.record("issue_refund", order_id=order_id,
                          amount_usd=amount_usd, reason=reason)
    return {"ok": True, "refund_id": f"REF-{1000 + len(ledger.entries)}", **entry}

print("wired: get_order, get_customer, search_policy, calc (read) "
      "+ issue_refund (risky, approval-gated)")

## The approval gate is a first-class pause

Chapter 08 wrapped the risky tools in `require_approval` so money moved only after a human said yes. Pydantic AI ships that as one flag. Because `issue_refund` is `requires_approval=True`, when the model calls it the run does **not** execute it — it stops and returns a `DeferredToolRequests` describing the pending call. That is why `output_type` is the union `[Decision, DeferredToolRequests]`: a run can end either with a finished verdict *or* with a pause waiting for approval. Run the triage and watch it halt before the money moves.

In [ ]:
TICKET = ("Triage Larkspur ticket TKT-2205 (order ORD-7312, customer CUST-07, sku "
          "LK-1016, qty 1): 'I opened the box and used the Torrent boots one evening "
          "indoors, they pinch at the toes. Repacked with tags. Refund my original "
          "payment method.'")

paused = await agent.run(TICKET)                       # multi-step loop, then the gate
print("run output type:", type(paused.output).__name__)
pending = paused.output.approvals[0]
print("pending tool  :", pending.tool_name)
print("proposed args :", pending.args)
print("ledger entries:", len(ledger.entries), "(gate held -- no money moved yet)")

> **What you should see:** the run stops with output type `DeferredToolRequests`, not `Decision`. The model read the order, customer, and policy, computed the 10% fee, and *asked* to call `issue_refund` for `170.99` on `ORD-7312` — but `requires_approval=True` turned that call into a pause instead of a side effect. The `Ledger` still reads `0`: nothing was refunded. This is chapter 08's approval gate, handed to you by the framework.

In [ ]:
from pydantic_ai import DeferredToolResults, ToolApproved

# A human reviews the pending call and approves (ch08's approver returns True).
approvals = {c.tool_call_id: ToolApproved() for c in paused.output.approvals}
resumed = await agent.run(message_history=paused.all_messages(),
                          deferred_tool_results=DeferredToolResults(approvals=approvals))

verdict = resumed.output
print("final output type:", type(verdict).__name__)
print("verdict          :", verdict)
print("ledger entries   :", len(ledger.entries), "(the approved write happened)")
print("matches gold     :", verdict.model_dump() == GOLD)

> **What you should see:** resuming with the approval turns the pending call into a real one — `issue_refund` runs, the `Ledger` ticks to `1`, and the agent returns a **validated `Decision`**: `partial_refund | pol-restocking | 170.99`. `verdict` is a Pydantic instance, not a string, so `verdict.refund_usd` is already a float with no parsing. `matches gold` is `True`: the framework landed on the same economics as `shoplab.rules.decide` (`$189.99 x 0.90`), the invariant every appendix in this set shares.

## Machinery map: PydanticAI feature to the part you built

Line them up and the framework stops being magic. Every PydanticAI concept here maps to a piece of machinery you built by hand in Parts 1-3.

| PydanticAI concept | Your hand-built equivalent | Built in |
|---|---|---|
| `output_type=Decision` (a Pydantic `BaseModel`) | `parse_json_loose` coaxing + validating a decision dict from model text | ch01 |
| the returned `verdict` object (typed, no parsing) | hand-checking the parsed dict's keys and types | ch01 |
| `OpenAIChatModel` + `OpenRouterProvider` | `shoplab.llm.complete` wrapping the model boundary | ch01 |
| `Agent(model, ...)` and its run loop | `run_agent`'s while-loop: model call -> tool call -> observe, repeat | ch02 |
| `@agent.tool_plain` (docstring -> desc, hints -> schema) | `to_openai_tools` building the function-call schema by hand | ch02 |
| `requires_approval=True` + `DeferredToolRequests` | `require_approval` wrapping a risky tool in a yes/no gate | ch08 |
| `DeferredToolResults(approvals=...)` to resume | the approver returning `True` and the loop continuing | ch08 |

## The honest trade

Pydantic AI's bet is types. You never wrote a parser here: `output_type=Decision` made the framework fill and validate the shape, and `verdict.refund_usd` came back a float you can bank on — the chapter 01 boundary, enforced by the library instead of by your own `parse_json_loose`. The approval gate is just as lean: one `requires_approval=True` flag reproduced chapter 08's pause-inspect-resume, and the pending call plus the whole message history come back as data you can serialize and reload in another process.

The cost is the usual framework tax, in a typed accent. The typed output is a *contract you now maintain* — change the cascade and the `Decision` schema, the instructions, and the resume plumbing all move together. The approval flow is two `run` calls and a `DeferredToolResults` round-trip, more ceremony than a plain `if approved:`. And the seam moved, exactly as chapters 11 and 16 warned: these calls go through Pydantic AI's own OpenAI client, not `shoplab.llm.complete` or LiteLLM, so the cost `LEDGER` and the disk cache from earlier chapters no longer see them. You are buying validation and resumability; know that you handed over the seam to get them.